<a href="https://colab.research.google.com/github/peteparker123/Bert-from-scratch-for-sentiment-analysis/blob/main/enhanced_bert_with_classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### import the modules

In [ ]:
import pandas as pd
import re
import torch
import torch.nn as nn
import math
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

### load and preprocess the data

In [ ]:
# Load and preprocess data
data = pd.read_csv("/content/train.csv", encoding='latin1')
data_clean = data.dropna(subset=["text", "sentiment"])

x = data_clean["text"].astype(str)
y = data_clean["sentiment"]

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
x_clean = x.map(preprocess_text)
# Build vocabulary
all_text = " ".join(x_clean) #convert to str
words = all_text.split() #split the sentences
unique_words = sorted(set(words)) #extract unique words
unique_words = ['<PAD>', '<UNK>'] + unique_words

#word-index mapping
stoi = {ch: i for i, ch in enumerate(unique_words)}
itos = {i: ch for i, ch in enumerate(unique_words)}

In [ ]:
BLOCK_SIZE = 24
PAD_IDX = stoi['<PAD>'] #used for embedding layers

def encode(s):
    words = s.split()
    return [stoi.get(w, stoi['<UNK>']) for w in words]  #check the word is there if not then use unk index

def encode_pad(s):
    encoded = encode(s)
    encoded = encoded[:BLOCK_SIZE] #trunnkation
    encoded += [stoi['<PAD>']] * (BLOCK_SIZE - len(encoded))  #padding
    return encoded

# Encode labels
le = LabelEncoder()
y_data = le.fit_transform(y)

In [ ]:
encode("how are you")

[10868, 1134, 26503]

### build custom dataset

In [ ]:
# Custom Dataset
class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels[idx]
        encoded = encode_pad(text)
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [ ]:
# Create dataset and dataloader
dataset = SentimentDataset(x_clean, y_data)
BATCH_SIZE = 32
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Model components
vocab_size = len(stoi)
embedding_dim = 64
num_classes = len(set(y_data))

### build model architecture

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.position_encoding = PositionalEncoding(embedding_dim, BLOCK_SIZE)

    def forward(self, x):
        x = self.embedding(x)
        x = self.position_encoding(x)
        return x


In [ ]:
input_embedding = InputEmbedding(vocab_size, embedding_dim)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def scaled_dot_product(self, q, k, v, mask):
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        return torch.matmul(attn, v)

    def forward(self, x, input_ids):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        mask = (input_ids != PAD_IDX).unsqueeze(1)
        return self.scaled_dot_product(q, k, v, mask)


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.norm = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        return self.dropout(self.norm(x))


In [ ]:
class FFN(nn.Module):
    def __init__(self, embedding_dim, ffn_dim):
        super().__init__()
        self.l1 = nn.Linear(embedding_dim, ffn_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.l2 = nn.Linear(ffn_dim, embedding_dim)

    def forward(self, x):
        x = self.l1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.l2(x)
        return x


In [ ]:
class Bert(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        self.embedding = InputEmbedding(vocab_size, embedding_dim)
        self.attention = SelfAttention(embedding_dim)
        self.norm1 = LayerNorm(embedding_dim)

        self.ffn = FFN(embedding_dim, 2 * embedding_dim)
        self.norm2 = LayerNorm(embedding_dim)

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, input_ids):
      x = self.embedding(input_ids)

      # Self-attention + residual + norm
      attn_out = self.attention(x, input_ids)
      x = self.norm1(x + attn_out)

      # FFN + residual + norm
      ffn_out = self.ffn(x)
      x = self.norm2(x + ffn_out)

      # Mean pooling
      mask = (input_ids != PAD_IDX).unsqueeze(-1)
      mask_sum = mask.sum(dim=1).clamp(min=1)
      x = (x * mask).sum(dim=1) / mask_sum

      return self.classifier(x)



In [ ]:
#model = Bert(vocab_size, embedding_dim, num_classes)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3,weight_decay=0.01)
criterion = nn.CrossEntropyLoss()


In [ ]:
from torch.utils.data import random_split

val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (input_ids, labels) in enumerate(dataloader):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        # 1. Forward pass
        logits = model(input_ids)

        # 2. Loss
        loss = criterion(logits, labels)

        # 3. Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_loss:.4f}")


Epoch [1/10] - Loss: 0.9270
Epoch [2/10] - Loss: 0.8232
Epoch [3/10] - Loss: 0.7712
Epoch [4/10] - Loss: 0.7213
Epoch [5/10] - Loss: 0.6684
Epoch [6/10] - Loss: 0.6170
Epoch [7/10] - Loss: 0.5727
Epoch [8/10] - Loss: 0.5289
Epoch [9/10] - Loss: 0.4830
Epoch [10/10] - Loss: 0.4425


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

EPOCHS = 10
patience = 2               # stop after 2 bad epochs
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    # -------- TRAIN --------
    model.train()
    train_loss = 0.0

    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        # 🔒 Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(dataloader)

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for input_ids, labels in val_dataloader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_dataloader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"- Train Loss: {train_loss:.4f} "
        f"- Val Loss: {val_loss:.4f}"
    )

    # -------- EARLY STOPPING --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        print(f"⚠️ No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("🛑 Early stopping triggered")
        break


Epoch [1/10] - Train Loss: 0.9564 - Val Loss: 0.8440
Epoch [2/10] - Train Loss: 0.8098 - Val Loss: 0.7994
Epoch [3/10] - Train Loss: 0.7302 - Val Loss: 0.7873
Epoch [4/10] - Train Loss: 0.6553 - Val Loss: 0.7749
Epoch [5/10] - Train Loss: 0.5913 - Val Loss: 0.7880
⚠️ No improvement (1/2)
Epoch [6/10] - Train Loss: 0.5310 - Val Loss: 0.8323
⚠️ No improvement (2/2)
🛑 Early stopping triggered


In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()


Bert(
  (embedding): InputEmbedding(
    (embedding): Embedding(26706, 64, padding_idx=0)
    (position_encoding): PositionalEncoding()
  )
  (attention): SelfAttention(
    (w_q): Linear(in_features=64, out_features=64, bias=True)
    (w_k): Linear(in_features=64, out_features=64, bias=True)
    (w_v): Linear(in_features=64, out_features=64, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm(
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (ffn): FFN(
    (l1): Linear(in_features=64, out_features=128, bias=True)
    (act): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
    (l2): Linear(in_features=128, out_features=64, bias=True)
  )
  (norm2): LayerNorm(
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (classifier): Linear(in_features=64, out_features=3, bias=True)
)

In [ ]:
def predict_multiple_sentiments(sentences):
    """Predict sentiment for multiple sentences"""
    model.eval()

    all_predictions = []

    with torch.no_grad():
        for sentence in sentences:
            # Preprocess each sentence
            processed = preprocess_text(sentence)
            encoded = encode_pad(processed)

            # Convert to tensor
            input_ids = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)

            # Get prediction
            logits = model(input_ids)
            probs = torch.softmax(logits, dim=-1)
            pred_id = torch.argmax(probs, dim=-1).item()

            # Get label and confidence
            label = le.inverse_transform([pred_id])[0]
            confidence = probs[0, pred_id].item()

            all_predictions.append((sentence, label, confidence))

    return all_predictions

# Load model
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

# Test multiple sentences
sentences = [
    "The product quality is really good and I am satisfied",
    "This is the worst purchase I've ever made",
    "It's okay, nothing special",
    "Absolutely fantastic! Would buy again!",
    "Terrible service and poor quality product"
]

results = predict_multiple_sentiments(sentences)

print("\nSentiment Analysis Results:")
print("=" * 60)
for sentence, label, confidence in results:
    print(f"Text: {sentence[:50]}...")
    print(f"Sentiment: {label} (Confidence: {confidence:.2%})")
    print("-" * 60)


Sentiment Analysis Results:
Text: The product quality is really good and I am satisf...
Sentiment: positive (Confidence: 79.38%)
------------------------------------------------------------
Text: This is the worst purchase I've ever made...
Sentiment: negative (Confidence: 96.81%)
------------------------------------------------------------
Text: It's okay, nothing special...
Sentiment: neutral (Confidence: 50.58%)
------------------------------------------------------------
Text: Absolutely fantastic! Would buy again!...
Sentiment: positive (Confidence: 93.85%)
------------------------------------------------------------
Text: Terrible service and poor quality product...
Sentiment: negative (Confidence: 98.20%)
------------------------------------------------------------


### build multihead attention

In [ ]:
num_heads = 2

In [ ]:
embedding_dim % num_heads == 0
embedding_dim = 128

head_dim = 64


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)

        self.w_o = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, input_ids):
        B, T, D = x.shape

        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)

        # split into heads
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        mask = (input_ids != PAD_IDX).unsqueeze(1).unsqueeze(2)
        scores = scores.masked_fill(mask == 0, -1e4)

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        # merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)

        return self.w_o(out)


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        self.attn = MultiHeadSelfAttention(embedding_dim, num_heads)
        self.norm1 = nn.LayerNorm(embedding_dim)

        self.ffn = FFN(embedding_dim, 4 * embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x, input_ids):
        x = self.norm1(x + self.attn(x, input_ids))
        x = self.norm2(x + self.ffn(x))
        return x


In [ ]:
class Bert(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes, num_heads=2, num_layers=2):
        super().__init__()

        self.embedding = InputEmbedding(vocab_size, embedding_dim)

        self.encoders = nn.ModuleList(
            [EncoderBlock(embedding_dim, num_heads) for _ in range(num_layers)]
        )

        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)

        for encoder in self.encoders:
            x = encoder(x, input_ids)

        # mean pooling
        mask = (input_ids != PAD_IDX).unsqueeze(-1)
        mask_sum = mask.sum(dim=1).clamp(min=1)
        x = (x * mask).sum(dim=1) / mask_sum

        return self.classifier(x)


In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Bert(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_classes=num_classes,
    num_heads=2,
    num_layers=2
).to(device)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

EPOCHS = 15
patience = 2
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    # -------- TRAIN --------
    model.train()
    train_loss = 0.0

    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        # 🔒 Gradient clipping (VERY IMPORTANT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(dataloader)

    # -------- VALIDATION --------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for input_ids, labels in val_dataloader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_dataloader)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"- Train Loss: {train_loss:.4f} "
        f"- Val Loss: {val_loss:.4f}"
    )

    # -------- EARLY STOPPING --------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        print(f"⚠️ No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("🛑 Early stopping triggered")
        break


Epoch [1/15] - Train Loss: 0.8733 - Val Loss: 0.7674
Epoch [2/15] - Train Loss: 0.7320 - Val Loss: 0.7772
⚠️ No improvement (1/2)
Epoch [3/15] - Train Loss: 0.6523 - Val Loss: 0.7530
Epoch [4/15] - Train Loss: 0.5728 - Val Loss: 0.7546
⚠️ No improvement (1/2)
Epoch [5/15] - Train Loss: 0.4975 - Val Loss: 0.8223
⚠️ No improvement (2/2)
🛑 Early stopping triggered


In [ ]:
def predict_multiple_sentiments(sentences):
    model.eval()

    # preprocess + encode all sentences
    encoded_sentences = [
        encode_pad(preprocess_text(sentence)) for sentence in sentences
    ]

    input_ids = torch.tensor(encoded_sentences, dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_ids)
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(probs, dim=-1)

    results = []
    for i, sentence in enumerate(sentences):
        label = le.inverse_transform([pred_ids[i].item()])[0]
        confidence = probs[i, pred_ids[i]].item()
        results.append((sentence, label, confidence))

    return results


In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()




Bert(
  (embedding): InputEmbedding(
    (embedding): Embedding(26706, 128, padding_idx=0)
    (position_encoding): PositionalEncoding()
  )
  (encoders): ModuleList(
    (0-1): 2 x EncoderBlock(
      (attn): MultiHeadSelfAttention(
        (w_q): Linear(in_features=128, out_features=128, bias=True)
        (w_k): Linear(in_features=128, out_features=128, bias=True)
        (w_v): Linear(in_features=128, out_features=128, bias=True)
        (w_o): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): FFN(
        (l1): Linear(in_features=128, out_features=512, bias=True)
        (act): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
        (l2): Linear(in_features=512, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (classifier): Linear(in_features=128, out_features=3, bi

In [ ]:
sentences = [
    "The product quality is really good and I am satisfied",
    "This is the worst purchase I've ever made",
    "It's okay, nothing special",
    "Absolutely fantastic! Would buy again!",
    "Terrible service and poor quality product"
]

results = predict_multiple_sentiments(sentences)

print("\nSentiment Analysis Results:")
print("=" * 60)
for sentence, label, confidence in results:
    print(f"Text: {sentence}")
    print(f"Sentiment: {label} (Confidence: {confidence:.2%})")
    print("-" * 60)


Sentiment Analysis Results:
Text: The product quality is really good and I am satisfied
Sentiment: positive (Confidence: 92.82%)
------------------------------------------------------------
Text: This is the worst purchase I've ever made
Sentiment: negative (Confidence: 95.75%)
------------------------------------------------------------
Text: It's okay, nothing special
Sentiment: positive (Confidence: 93.07%)
------------------------------------------------------------
Text: Absolutely fantastic! Would buy again!
Sentiment: positive (Confidence: 89.63%)
------------------------------------------------------------
Text: Terrible service and poor quality product
Sentiment: negative (Confidence: 94.34%)
------------------------------------------------------------
